# Data mentioned in the text of the Background & Summary section
Every number cited in *Background & Summary*, recomputed from `cultura_enriched_v3.2`, the Cross-Verified Database and Cliopatria.

In [1]:
import json
from pathlib import Path

import duckdb

ROOT = Path.cwd().resolve()
while not (ROOT / '.git').exists():
    ROOT = ROOT.parent

DB_PATH = ROOT / 'data' / 'cultura' / 'cultura_enriched_v3.2.duckdb'  # read only
CROSS_VERIFIED = ROOT / 'data' / 'similar_databases' / 'cross_verified.duckdb'
CLIOPATRIA = ROOT / 'data' / 'cliopatria_data' / 'cliopatria_V2' / 'cliopatria_polities_only_v3.geojson'
CROSS_VERIFIED_EDITIONS = ['en', 'fr', 'de', 'it', 'es', 'pt', 'sv']

connection = duckdb.connect(str(DB_PATH), read_only=True)

## Size of Cultura and of the Cross-Verified Database (paragraphs 2 and 5)

In [2]:
cultura_total = connection.sql('SELECT count(*) FROM individual_enriched').fetchone()[0]
cross_verified_total = duckdb.connect(str(CROSS_VERIFIED), read_only=True).sql('SELECT count(*) FROM individuals').fetchone()[0]
print(f'Cultura: {cultura_total:,} individuals')
print(f'Cross-Verified: {cross_verified_total:,} individuals')
print(f'Ratio: {cultura_total / cross_verified_total:.1f}')

Cultura: 13,002,897 individuals
Cross-Verified: 2,291,817 individuals
Ratio: 5.7


## Individuals outside the seven Wikipedia editions of the Cross-Verified Database (paragraph 3)
Individuals of Cultura with no article in English, French, German, Italian, Spanish, Portuguese or Swedish Wikipedia, and those with no Wikipedia article at all (Commons and other Wikimedia projects are not Wikipedia articles).

In [3]:
sites = ', '.join(f"'https://{edition}.wikipedia.org'" for edition in CROSS_VERIFIED_EDITIONS)
in_seven = connection.sql(f'SELECT count(DISTINCT qid) FROM individual_sitelink WHERE site_url IN ({sites})').fetchone()[0]
in_any_wikipedia = connection.sql("SELECT count(DISTINCT qid) FROM individual_sitelink WHERE site_url LIKE '%.wikipedia.org'").fetchone()[0]
print(f'In at least one of the seven editions: {in_seven:,} ({in_seven / cultura_total:.1%})')
print(f'In another Wikipedia only: {in_any_wikipedia - in_seven:,} ({(in_any_wikipedia - in_seven) / cultura_total:.1%})')
print(f'In no Wikipedia: {cultura_total - in_any_wikipedia:,} ({(cultura_total - in_any_wikipedia) / cultura_total:.1%})')

In at least one of the seven editions: 3,161,340 (24.3%)
In another Wikipedia only: 1,582,633 (12.2%)
In no Wikipedia: 8,258,924 (63.5%)


## Cliopatria (paragraph 3)

In [4]:
features = json.loads(CLIOPATRIA.read_text())['features']
names = {feature['properties']['Name'] for feature in features}
years = [year for feature in features for year in (feature['properties']['FromYear'], feature['properties']['ToYear'])]
print(f'{len(names):,} polities, {len(features):,} territories, {min(years)} to {max(years)}')

1,633 polities, 13,755 territories, -3400 to 2024


## Individuals linked to a polity (paragraph 3)

In [5]:
with_polity = connection.sql('SELECT count(*) FROM individual_enriched WHERE main_polity IS NOT NULL').fetchone()[0]
print(f'{with_polity:,} individuals with a main polity ({with_polity / cultura_total:.1%})')

3,546,784 individuals with a main polity (27.3%)
